# Pruebas archivo .f_a

## pruebas con el .csv

In [1]:
import os
import glob
import shutil
import pandas as pd
import numpy as np

In [2]:
ruta_base = "../data/data_bis_advanced"

for raiz, dirs, files in os.walk(ruta_base):
    print(f"\nCarpeta: {raiz}")
    print("Subcarpetas:", dirs)
    print("Archivos:", files)


Carpeta: ../data/data_bis_advanced
Subcarpetas: ['M-TA6m-03041035', 'M-TA6m-03041035_2']
Archivos: []

Carpeta: ../data/data_bis_advanced\M-TA6m-03041035
Subcarpetas: ['BIS_TA6m_04032026103520', 'DH03041035', 'DSA_TA6m_04032026103520']
Archivos: []

Carpeta: ../data/data_bis_advanced\M-TA6m-03041035\BIS_TA6m_04032026103520
Subcarpetas: []
Archivos: ['BIS_TA6m_20260304_1-1.pdf']

Carpeta: ../data/data_bis_advanced\M-TA6m-03041035\DH03041035
Subcarpetas: []
Archivos: ['L03041035.ara', 'L03041035.csv', 'L03041035.e_a', 'L03041035.f_a', 'L03041035.h_a', 'L03041035.m_a', 'L03041035.o_a', 'L03041035.r2a', 'L03041035.spa', 'L03041035.t_a', 'L03041035_proc.csv']

Carpeta: ../data/data_bis_advanced\M-TA6m-03041035\DSA_TA6m_04032026103520
Subcarpetas: []
Archivos: ['DSA_TA6m_20260304_1-1.pdf']

Carpeta: ../data/data_bis_advanced\M-TA6m-03041035_2
Subcarpetas: ['BIS_TA6m_04032026103520', 'DH03041035', 'DSA_TA6m_04032026103520']
Archivos: []

Carpeta: ../data/data_bis_advanced\M-TA6m-03041035_2\B

In [ ]:
def localizar_archivos_fa(ruta_raiz):
    """
    Busca archivos .f_a dentro de subcarpetas cuyo nombre empiece por DH.
    """
    patron = os.path.join(ruta_raiz, "**", "DH*", "*.f_a")
    archivos = glob.glob(patron, recursive=True)
    print(f"Se han encontrado {len(archivos)} archivos .f_a")
    return archivos


def clonar_a_csv(ruta_fa):
    """
    Copia el archivo .f_a y crea una versión .csv con el mismo contenido.
    """
    ruta_csv = os.path.splitext(ruta_fa)[0] + ".csv"
    shutil.copy2(ruta_fa, ruta_csv)
    print(f"Archivo copiado como: {ruta_csv}")
    return ruta_csv


def procesar_datos_csv(ruta_csv):
    """
    Procesa el archivo BIS:
    - Lee el archivo usando '|' como separador principal
    - Se queda con las columnas Time y Spectra
    - Divide Spectra en 60 columnas
    - Convierte Time a datetime
    - Convierte Spectra a float
    - Divide los valores entre 100
    - Guarda un archivo procesado con sufijo _proc
    """
    try:
        # Leer archivo
        df = pd.read_csv(
            ruta_csv,
            sep="|",
            header=None,
            skiprows=2,
            engine="python"
        )

        # Eliminar columnas completamente vacías
        df = df.dropna(axis=1, how="all")

        # Quedarnos solo con las dos columnas reales
        df = df.iloc[:, :2]
        df.columns = ["Time", "Spectra"]

        # Dividir columna Spectra en varias columnas
        spectra = df["Spectra"].astype(str).str.split(",", expand=True)
        spectra.columns = [f"Spectra_{i+1}" for i in range(spectra.shape[1])]

        # Unir Time y espectros
        df_final = pd.concat([df[["Time"]], spectra], axis=1)

        # Convertir tipos
        df_final["Time"] = pd.to_datetime(
            df_final["Time"],
            format="%m/%d/%Y %H:%M:%S"
        )
        df_final.iloc[:, 1:] = df_final.iloc[:, 1:].astype(float) / 100

        # Guardar resultado
        base, ext = os.path.splitext(ruta_csv)
        ruta_salida = f"{base}_proc{ext}"
        df_final.to_csv(ruta_salida, index=False)

        print(f"Archivo procesado guardado en: {ruta_salida}")
        return df_final

    except Exception as e:
        print(f"Error en el procesamiento de {ruta_csv}: {e}")
        return None


def procesar_todos_los_archivos(ruta_base):
    """
    Ejecuta todo el flujo sobre todos los archivos .f_a encontrados.
    """
    archivos_fa = localizar_archivos_fa(ruta_base)

    resultados = {}

    for ruta_fa in archivos_fa:
        print(f"\nProcesando: {ruta_fa}")
        ruta_csv = clonar_a_csv(ruta_fa)
        df_final = procesar_datos_csv(ruta_csv)

        if df_final is not None:
            resultados[ruta_fa] = df_final

    return resultados

In [ ]:
resultados = procesar_todos_los_archivos(ruta_base)


In [ ]:
# probar ver como se ve finalmente el archivo

primer_archivo = list(resultados.keys())[0]
display(resultados[primer_archivo].head())
print(resultados[primer_archivo].shape)

In [ ]:
# el segundo era una copia del 1

seg_archivo = list(resultados.keys())[1]
display(resultados[seg_archivo].head())
print(resultados[seg_archivo].shape)